In [1]:
import torch
torch.cuda.empty_cache()

In [2]:
!pip install diffusers transformers gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.1/322.1 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 98.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 4.3 MB/s eta 0:00:00
  Attempting uninstall: markupsafe
    Found existing installation: MarkupSafe 3.0.2
    Uninstalling MarkupSafe-3.0.2:
      Successfully uninstalled MarkupSafe-3.0.2


In [3]:
pip install tensorflow tensorflow-hub

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import gradio as gr
from PIL import Image, ImageEnhance
import torch
from diffusers import StableDiffusionPipeline
import io
import os
import numpy as np
import tensorflow as tf
import tensorflow_hub as hub
import time
import tempfile
from deep_translator import GoogleTranslator  # More reliable translation library



# Available models
MODEL_OPTIONS = {
    "Stable Diffusion v1-4": "CompVis/stable-diffusion-v1-4",
    "Stable Diffusion v2-1": "stabilityai/stable-diffusion-2-1",
    "DreamShaper v7": "Lykon/dreamshaper-7"
}

# Detect the appropriate device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load TensorFlow style transfer model
style_transfer_model = hub.load('https://tfhub.dev/google/magenta/arbitrary-image-stylization-v1-256/2')

# Function to translate text to English
def translate_to_english(text):
    """Translates text from any detected language to English."""
    try:
        if not text or not text.strip():
            return text, "en"
        
        # First detect if the text is already in English
        # The library doesn't have a direct way to detect language, so we'll use a simple heuristic
        translator = GoogleTranslator(source='auto', target='en')
        translated = translator.translate(text)
        
        # If the translation is identical to the original (ignoring case and punctuation),
        # it's likely already in English
        if translated.lower().strip('.,?!') == text.lower().strip('.,?!'):
            return text, "en"
        
        # Get source language from the translator
        source_lang = translator.source
        if source_lang == 'auto':
            source_lang = 'unknown'
            
        return translated, source_lang
    except Exception as e:
        print(f"Translation error: {e}")
        return text, "unknown"  # Return original text if translation fails

# Function to load stable diffusion model
def load_model(model_id):
    pipe = StableDiffusionPipeline.from_pretrained(
        model_id, torch_dtype=torch.float16, use_auth_token=os.environ["HF_TOKEN"]
    )
    pipe.to(device)
    return pipe

# Dictionary to cache models
model_cache = {}

def generate_image(model_name, prompt):
    # Translate the prompt to English
    original_prompt = prompt
    translated_prompt, source_lang = translate_to_english(prompt)
    
    # Add a prefix to show the translation occurred (if it did)
    display_prompt = translated_prompt
    was_translated = original_prompt != translated_prompt
    
    model_id = MODEL_OPTIONS[model_name]
    if model_id not in model_cache:
        model_cache[model_id] = load_model(model_id)
    
    pipe = model_cache[model_id]
    generated_image = pipe(translated_prompt, guidance_scale=8.5, num_inference_steps=20).images[0]
    
    # Return the image and whether translation occurred
    return generated_image, was_translated, original_prompt, translated_prompt, source_lang

def enhance_image(image, sharpness, brightness, contrast, upscale):
    if image is None:
        return None
        
    image = Image.fromarray(image)
    image = ImageEnhance.Sharpness(image).enhance(sharpness)
    image = ImageEnhance.Brightness(image).enhance(brightness)
    image = ImageEnhance.Contrast(image).enhance(contrast)
    
    if upscale:
        new_size = (image.width * 2, image.height * 2)
        image = image.resize(new_size, Image.LANCZOS)
    
    return np.array(image)

def preprocess_image(image_path):
    """Preprocesses images for the style transfer model."""
    # If image is a file path string
    if isinstance(image_path, str):
        img = tf.io.read_file(image_path)
        img = tf.image.decode_image(img, channels=3)
    # If image is a PIL Image or numpy array
    else:
        if isinstance(image_path, Image.Image):
            img = np.array(image_path)
        else:
            img = image_path
        img = tf.convert_to_tensor(img)
    
    # Convert to float32 and add batch dimension
    img = tf.cast(img, tf.float32)
    img = img / 255.0
    if len(img.shape) == 3:
        img = tf.expand_dims(img, 0)
    
    return img

def apply_style_transfer(content_image, style_image, style_weight=1.0):
    """Apply neural style transfer to a content image using a style image."""
    if content_image is None or style_image is None:
        return None
    
    # Convert PIL images or numpy arrays to tensors
    content_tensor = preprocess_image(content_image)
    style_tensor = preprocess_image(style_image)
    
    # Ensure both images have 3 dimensions plus batch dimension
    if content_tensor.shape[-1] != 3 or style_tensor.shape[-1] != 3:
        raise ValueError("Both content and style images must have 3 color channels")
    
    # Apply style transfer
    stylized_image = style_transfer_model(tf.constant(content_tensor), 
                                          tf.constant(style_tensor))[0]
    
    # Convert back to numpy array
    stylized_image = stylized_image[0].numpy()
    stylized_image = (stylized_image * 255).astype(np.uint8)
    
    return stylized_image

# Proper image download function
def save_and_download_image(image, filename_prefix="image"):
    """Save image to a temporary file and return the path for download."""
    if image is None:
        return None
    
    # Convert to PIL Image if it's a numpy array
    if isinstance(image, np.ndarray):
        image = Image.fromarray(image)
    elif not isinstance(image, Image.Image):
        # If it's a PIL Image already, no conversion needed
        # If it's something else (which shouldn't happen), convert to numpy first
        image = Image.fromarray(np.array(image))
    
    # Create a temporary file
    temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".png")
    temp_filename = temp_file.name
    temp_file.close()
    
    # Save the image to the temporary file
    image.save(temp_filename)
    
    # Return the path to the temporary file
    return temp_filename

# Main Gradio interface
with gr.Blocks() as demo:
    gr.Markdown("## ✨ AI Image Generator with Neural Style Transfer 🎨")
    gr.Markdown("Generate an image, apply enhancements, or transform its style using neural networks.")
    gr.Markdown("✅ **Multilingual Support**: Enter prompts in any language - they'll be automatically translated!")
    
    # Store the current active image for download
    current_active_image = gr.State(value=None)
    
    with gr.Tab("Generate Image"):
        model_dropdown = gr.Dropdown(list(MODEL_OPTIONS.keys()), label="Choose a Model")
        prompt_input = gr.Textbox(label="Enter your prompt (any language):")
        generate_btn = gr.Button("🚀 Generate Image")
        image_output = gr.Image()
        translation_info = gr.Markdown(visible=False)
        download_generated_btn = gr.Button("📥 Download Generated Image")
    
    with gr.Tab("Enhance Image"):
        with gr.Row():
            enhance_input = gr.Image(label="Image to Enhance")
            enhanced_output = gr.Image(label="Enhanced Image")
        
        with gr.Row():
            sharpness_slider = gr.Slider(0.5, 3.0, 1.0, label="Sharpness")
            brightness_slider = gr.Slider(0.5, 2.0, 1.0, label="Brightness")
            contrast_slider = gr.Slider(0.5, 2.0, 1.0, label="Contrast")
            upscale_checkbox = gr.Checkbox(label="Enable Upscaling (2x)")
        
        enhance_btn = gr.Button("✨ Enhance Image")
        download_enhanced_btn = gr.Button("📥 Download Enhanced Image")
    
    with gr.Tab("Style Transfer"):
        with gr.Row():
            content_image_input = gr.Image(label="Content Image")
            style_image_input = gr.Image(label="Style Image")
            stylized_output = gr.Image(label="Stylized Result")
        
        style_weight_slider = gr.Slider(0.1, 2.0, 1.0, label="Style Weight")
        style_transfer_btn = gr.Button("🎨 Apply Style Transfer")
        download_stylized_btn = gr.Button("📥 Download Stylized Image")
        
        gr.Markdown("""
        ### How to use Style Transfer:
        1. Upload or generate a content image (what you want to transform)
        2. Upload a style image (the artistic style to apply)
        3. Adjust the style weight slider to control the strength of the style
        4. Click "Apply Style Transfer" to blend them together
        """)
    
    # Add cross-tab functionality
    with gr.Row():
        use_generated_as_content_btn = gr.Button("Use Generated Image as Content")
        use_generated_as_enhance_btn = gr.Button("Use Generated Image for Enhancement")
        use_enhanced_as_content_btn = gr.Button("Use Enhanced Image as Content")
    
    # Output component for file downloads
    download_file = gr.File(label="Downloaded Image")
    
    # Modified generate function to handle translation info
    def generate_with_translation_info(model_name, prompt):
        image, was_translated, original, translated, source_lang = generate_image(model_name, prompt)
        
        # Prepare translation info message
        if was_translated:
            info_text = f"**Translation Applied:**\n- Original ({source_lang}): \"{original}\"\n- Translated (en): \"{translated}\""
        else:
            info_text = "No translation needed (English detected)."
            
        return image, gr.update(value=info_text, visible=True)
    
    # Connect the components with their functions
    generate_btn.click(
        generate_with_translation_info, 
        inputs=[model_dropdown, prompt_input], 
        outputs=[image_output, translation_info]
    )
    
    enhance_btn.click(
        enhance_image, 
        inputs=[enhance_input, sharpness_slider, brightness_slider, contrast_slider, upscale_checkbox], 
        outputs=enhanced_output
    )
    
    style_transfer_btn.click(
        apply_style_transfer,
        inputs=[content_image_input, style_image_input, style_weight_slider],
        outputs=stylized_output
    )
    
    # Cross-tab connections
    use_generated_as_content_btn.click(
        lambda x: x, 
        inputs=image_output, 
        outputs=content_image_input
    )
    
    # Add connection for using generated image in enhance tab
    use_generated_as_enhance_btn.click(
        lambda x: x, 
        inputs=image_output, 
        outputs=enhance_input
    )
    
    use_enhanced_as_content_btn.click(
        lambda x: x, 
        inputs=enhanced_output, 
        outputs=content_image_input
    )
    
    # Download buttons for each tab
    download_generated_btn.click(
        save_and_download_image,
        inputs=[image_output],
        outputs=[download_file]
    )
    
    download_enhanced_btn.click(
        save_and_download_image,
        inputs=[enhanced_output],
        outputs=[download_file]
    )
    
    download_stylized_btn.click(
        save_and_download_image,
        inputs=[stylized_output],
        outputs=[download_file]
    )

demo.launch()

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

* Running on local URL:  http://127.0.0.1:7860
Kaggle notebooks require sharing enabled. Setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://1418db954c4573ce49.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
